# Tabela FIPE e ficha técnica de veículos: exploratory analysis

FIPE prices of Brazilian cars, motorcycles and trucks since 2020, specifications of 37,794 versions and vehicle × product matches. Dataset: [Kaggle](https://www.kaggle.com/datasets/caiquedourado/fipe-ficha-tecnica-veiculos) · [GitHub](https://github.com/CaiqueDourado/fipe-ficha-tecnica-veiculos) (methodology, data quality report, data dictionary).

Runs on Kaggle with the full files, or locally with the repository's `sample/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

KAGGLE = next((p.parent for p in Path("/kaggle/input").glob("**/marcas.parquet")), Path("/kaggle/input/none"))
def load(name):
    if KAGGLE.exists():
        return pd.read_parquet(KAGGLE / f"{name}.parquet")
    return pd.read_csv(Path("../sample") / f"{name}_sample.csv", dtype={"codigo_fipe": str})

marcas, modelos, precos = load("marcas"), load("modelos"), load("precos")
fichas, vp = load("fichas"), load("veiculo_produto")
precos["referencia"] = pd.to_datetime(precos.referencia)
precos["zero_km"] = precos.zero_km.astype(str).str.lower() == "true"
precos = precos.merge(modelos[["codigo_modelo", "tipo_veiculo"]], on="codigo_modelo")
print(f"{len(precos):,} prices, {precos.referencia.nunique()} months, {len(fichas):,} versions with specifications")

## Prices per month and vehicle type

In [ ]:
precos.groupby(["referencia", "tipo_veiculo"]).size().unstack().tail(8)

## Used vehicles in 2020–2023

The same FIPE code and model year, month by month, relative to January 2020.

In [ ]:
d = precos[(precos.fonte == "dump") & ~precos.zero_km].copy()
d["serie"] = d.codigo_fipe + "/" + d.ano_modelo.astype("Int64").astype(str) + "/" + d.combustivel
base = d[d.referencia == d.referencia.min()].drop_duplicates("serie").set_index("serie").valor
d = d[d.serie.isin(base.index)]
d["indice"] = 100 * d.valor / d.serie.map(base)
idx = d.groupby(["referencia", "tipo_veiculo"]).indice.median().unstack()
ax = idx.plot(figsize=(10, 4), title="Median FIPE price, January 2020 = 100")
ax.axhline(100, color="gray", linewidth=0.8, linestyle=":")
idx.iloc[-1].round(1)

## Zero km and depreciation

Share of the zero km price that a car keeps, by age, in the latest month.

In [ ]:
ref = precos.referencia.max()
last = precos[(precos.referencia == ref) & (precos.tipo_veiculo == "carro")]
zk = last[last.zero_km].drop_duplicates("codigo_fipe").set_index("codigo_fipe").valor
used = last[~last.zero_km & last.codigo_fipe.isin(zk.index)].copy()
used["idade"] = ref.year - used.ano_modelo
used["fracao"] = used.valor / used.codigo_fipe.map(zk)
used[used.idade.between(0, 15)].groupby("idade").fracao.agg(["median", "count"]).round(2)

## Specifications: which items became standard

Share of car versions with each item, by model year.

In [ ]:
carros = fichas[fichas.tipo_veiculo == "carro"]
itens = ["airbag_motorista", "freios_abs", "ar_condicionado", "direcao_eletrica", "entrada_usb", "kit_multimidia"]
share = carros.groupby("ano_modelo")[itens].mean().loc[2000:2023]
share.plot(figsize=(10, 4), title="Share of versions with the item")
share.tail(3).round(2)

## Specifications next to the FIPE price

In [ ]:
feb23 = precos[(precos.referencia == precos.referencia.max()) & ~precos.zero_km][["codigo_fipe", "ano_modelo", "valor"]]
j = carros.merge(feb23, on=["codigo_fipe", "ano_modelo"])
j["cv"] = j.potencia_cv.str.extract(r"(\d+)").astype(float)
j[["marca", "modelo", "versao", "ano_modelo", "potencia_cv", "valor"]].sort_values("valor").tail(5)

## Vehicle × product: read the warning level

`confianca` says how wide the matching key was: most pairs are `baixa` (the key matched 1,000 or more product groups).

In [ ]:
vp.confianca.value_counts(normalize=True).round(3)

In [ ]:
vp[vp.confianca == "alta"].merge(modelos[["codigo_modelo", "modelo"]], on="codigo_modelo").head(10)

## Text from the data

The repository's `textos/` module writes the "about" paragraph, the equipment and a FAQ of a version, and its FIPE price history, by rules (no language model: every sentence comes from a field).

In [ ]:
import sys, subprocess
try:
    import textos
except ImportError:
    if KAGGLE.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/CaiqueDourado/fipe-ficha-tecnica-veiculos"], check=True)
    else:
        sys.path.insert(0, "..")
from textos import Vehicle, history

ficha = fichas[(fichas.codigo_fipe == "003268-9") & (fichas.ano_modelo == 2012)]
ficha = ficha.iloc[0] if len(ficha) else fichas[fichas.tipo_veiculo == "carro"].iloc[0]
v = Vehicle(ficha.to_dict())
print(v.about(), *v.equipment().values(), sep="\n\n")

In [ ]:
serie = precos[(precos.codigo_fipe == v.get("codigo_fipe")) & (precos.ano_modelo == v.get("ano_modelo"))]
print(history(zip(serie.referencia, serie.valor), v.title))
v.faq(include_no=False)[:5]